# 04 · Clientes e Vulto+ da Vulto: raio-x

Fontes: `clientes_clientes.csv`, `clientes_antigos.csv` e `clientes_resgates.csv` (abas de `clientes_fidelidade_FINAL_v3 (1).xlsx`), `banco_pontos_mov.csv` e `banco_compras.csv` (do `vulto.db`).

Objetivo: **entender** os clientes e os pontos, sem decidir nada. As mesmas 4 perguntas.

**Regras do programa** (aba "LEIA ISSO" da planilha, jun/2025): R$ 1 = 1 ponto na loja física; site e app valem **dobro**; Ouro ganha **+50%**; **100 pontos = R$ 5** (era R$ 10 até fev/2025); pontos **vencem em 12 meses** (18 para Ouro, "ainda não tá no sistema"); resgate mínimo de 100 pontos; não pode ficar negativo ("o sistema deixa"); pontos só com CPF. **A coluna `Pontos` da aba Clientes foi congelada em 01/01/2025**: depois disso os pontos ficam no banco (`pontos_mov`). Algumas lojas continuaram anotando resgates na planilha. O banco ainda usa R$ 0,10 por ponto (`config.valor_ponto`).

> Antes de commitar: *Kernel → Restart & Clear Output* (as tabelas têm CPF, e-mail e telefone).

In [ ]:
from pathlib import Path

import pandas as pd

# raiz do repositório, de onde quer que o notebook tenha sido aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "migracao_vulto").is_dir())
STAGING = RAIZ / "staging_vulto"

# mostra todas as colunas nas tabelas
pd.set_option("display.max_columns", None)

In [ ]:
clientes = pd.read_csv(STAGING / "clientes_clientes.csv", dtype=str)
antigos = pd.read_csv(STAGING / "clientes_antigos.csv", dtype=str)
resgates = pd.read_csv(STAGING / "clientes_resgates.csv", dtype=str)
pontos = pd.read_csv(STAGING / "banco_pontos_mov.csv", dtype=str)
compras = pd.read_csv(STAGING / "banco_compras.csv", dtype=str)
pontos["q"], compras["v"] = pd.to_numeric(pontos["qtd"]), pd.to_numeric(compras["valor"])

## 1. O que é cada linha?

- `clientes`: um cadastro do Vulto+ (não uma pessoa: a mesma pessoa pode ter vários cadastros).
- `antigos`: um cliente do programa de 2023.
- `resgates`: um resgate anotado na planilha pela loja.
- `pontos`: um movimento de pontos no banco: C crédito, D resgate, E estorno, AJ ajuste.
- `compras`: uma compra que pode gerar pontos (o "lado fidelidade" da venda).

In [ ]:
for nome, tabela in [("clientes", clientes), ("antigos", antigos), ("resgates", resgates), ("pontos", pontos), ("compras", compras)]:
    print(nome, tabela.shape)
# a mesma pessoa em mais de um cadastro
print("CPFs com 2 ou mais cadastros:", clientes["cpf"].dropna().value_counts().gt(1).sum(),
      "| e-mails repetidos:", clientes["E-mail"].dropna().str.lower().duplicated().sum())
clientes[clientes["cpf"].isin(clientes["cpf"].dropna().value_counts().head(2).index)].sort_values("cpf")

## 2. O que significa cada coluna?

In [ ]:
# Pontos da planilha: vários formatos (9 = qualquer dígito)
display(clientes["Pontos"].dropna().str.replace(r"\d", "9", regex=True).value_counts().head(8))
for coluna in ["categoria", "aceita_msg", "loja_cadastro"]:
    print(coluna, "|", clientes[coluna].value_counts(dropna=False).to_dict())
cadastro = pd.to_datetime(clientes["Data Cadastro"], errors="coerce")
print("cadastrados a partir de 2025:", (cadastro >= "2025-01-01").sum(),
      "| desses com Pontos na planilha (que foi congelada em 01/01/2025):", ((cadastro >= "2025-01-01") & clientes["Pontos"].notna() & (clientes["Pontos"] != "0")).sum())

In [ ]:
# pontos no banco por tipo, e o período
print("período:", pontos["criado_em"].min(), "a", pontos["criado_em"].max())
pontos.groupby("tipo")["q"].agg(["count", "sum"])

## 3. Que problemas aparecem?

In [ ]:
# pontos creditados / valor da compra: deveria ser 1 na loja física e 2 no site e no app
creditos = pontos[pontos["tipo"] == "C"].assign(compra=lambda d: d["compra_id"].str.removesuffix(".0")).merge(
    compras[["id", "v", "origem", "canal"]], left_on="compra", right_on="id", suffixes=("", "_compra"))
creditos["pontos_por_real"] = (creditos["q"] / creditos["v"]).round(2)
# no app o valor está em centavos: 0,02 = 2 pontos por real; 0,01 = 1 ponto por real
pd.crosstab(creditos["canal"].str.lower(), creditos["pontos_por_real"])

In [ ]:
# bug do app: pontos em dobro sobre o valor em CENTAVOS (pontos_por_real = 2 com valor acima de 5 mil)
inflados = creditos[(creditos["origem"] == "app") & (creditos["pontos_por_real"] == 2) & (creditos["v"] > 5000)]
ajustes_bug = pontos[(pontos["tipo"] == "AJ") & pontos["obs"].fillna("").str.contains("bug")]
print("créditos inflados:", len(inflados), "| pontos:", inflados["q"].sum(), "| quando:", inflados["criado_em"].min()[:10], "a", inflados["criado_em"].max()[:10])
print("ajustes 'correcao bug app':", len(ajustes_bug), "| pontos:", ajustes_bug["q"].sum(), "| quando:", ajustes_bug["criado_em"].min()[:10], "a", ajustes_bug["criado_em"].max()[:10])
sem_ajuste = inflados[~inflados["cpf"].isin(ajustes_bug["cpf"])]
print("inflados SEM correção:", sem_ajuste["cpf"].nunique(), "clientes,", sem_ajuste["q"].sum(), "pontos")
print("ajustados SEM crédito inflado:", (~ajustes_bug["cpf"].isin(inflados["cpf"])).sum(), "clientes,", ajustes_bug.loc[~ajustes_bug["cpf"].isin(inflados["cpf"]), "q"].sum(), "pontos")

In [ ]:
# resgates anotados na planilha a partir de 2025: também estão no banco?
resgates["dia"] = pd.to_datetime(resgates["data"])
resgates_banco = pontos[pontos["tipo"] == "D"].assign(dia=lambda d: pd.to_datetime(d["criado_em"]).dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None).dt.normalize())
recentes = resgates[(resgates["dia"] >= "2025-01-01") & resgates["cliente_cpf"].notna()].copy()
recentes["no_banco"] = [((resgates_banco["cpf"] == cpf) & (resgates_banco["q"] == float(q)) & ((resgates_banco["dia"] - dia).abs() <= pd.Timedelta(days=3))).any()
                        for cpf, q, dia in zip(recentes["cliente_cpf"], recentes["pontos usados"], recentes["dia"])]
print("resgates da planilha desde 2025:", (resgates["dia"] >= "2025-01-01").sum(), "| com CPF:", len(recentes), "| sem CPF (só nome):", ((resgates["dia"] >= "2025-01-01") & resgates["cliente_cpf"].isna()).sum())
recentes.groupby("loja")["no_banco"].agg(["sum", "count"])

In [ ]:
# clientes antigos (programa de 2023): quantos também estão na aba Clientes
antigos.assign(tambem_em_clientes=antigos["CPF/CNPJ"].isin(clientes["cpf"])).groupby(antigos["STATUS"].fillna("(vazio)"))["tambem_em_clientes"].agg(["sum", "count"])

## 4. Como se liga às outras fontes?

In [ ]:
print("compras por origem e loja:")
display(pd.crosstab(compras["origem"], compras["loja_id"]))
sem_pontos = compras[~compras["id"].isin(pontos["compra_id"].dropna().str.removesuffix(".0"))]
print("compras sem crédito de pontos:", len(sem_pontos), "| por tipo de cliente:", sem_pontos["cliente_tipo"].fillna("(vazio)").value_counts().to_dict())
print("CPFs com pontos no banco que não estão na planilha:", pontos.loc[~pontos["cpf"].isin(clientes["cpf"]), "cpf"].nunique())

## Desafios da revisão

In [ ]:
# nos CPFs com 2 ou mais cadastros: os dois têm pontos?
def pontos_planilha(valor):  # "2.203", "1389 pts", "15,0k" → número (só para medir)
    if pd.isna(valor):
        return 0.0
    s = str(valor).lower().replace("pts", "").strip()
    return float(s[:-1].replace(",", ".")) * 1000 if s.endswith("k") else float(s.replace(".", ""))

clientes["pontos_num"] = clientes["Pontos"].map(pontos_planilha)
duplicados = clientes[clientes["cpf"].notna() & clientes["cpf"].duplicated(keep=False)]
com_pontos = duplicados.groupby("cpf")["pontos_num"].apply(lambda s: (s > 0).sum())
print("CPFs duplicados:", len(com_pontos), "| nenhum cadastro com pontos:", (com_pontos == 0).sum(),
      "| só 1 com pontos:", (com_pontos == 1).sum(), "| 2 ou mais com pontos:", (com_pontos >= 2).sum())
print("pontos nos cadastros extras (além do maior de cada CPF):", duplicados.groupby("cpf")["pontos_num"].apply(lambda s: s.sum() - s.max()).sum())

In [ ]:
# os CPFs só no banco: clientes novos ou CPF digitado errado (1 dígito diferente de um CPF da planilha)?
inicio = pontos[pontos["cpf"].notna() & ~pontos["cpf"].isin(clientes["cpf"])].groupby("cpf")["criado_em"].min()
cpfs_planilha = list(clientes["cpf"].dropna().unique())
um_digito = sum(any(sum(a != b for a, b in zip(cpf, outro)) == 1 for outro in cpfs_planilha) for cpf in inicio.index)
print("CPFs só no banco:", len(inicio), "| começaram em 2025:", (inicio < "2026").sum(), "| em 2026:", (inicio >= "2026").sum(),
      "| diferem só 1 dígito de um CPF da planilha:", um_digito)

## Saldo inicial (K5)

De onde vem o saldo de cada cliente: a planilha congelada em 01/01/2025, o banco depois disso, e as duas abas da planilha (antiga e nova).

In [ ]:
# o banco continua o saldo da planilha? clientes que ficam negativos com e sem o saldo inicial
pontos["mov"] = pontos["q"].where(pontos["tipo"] != "D", -pontos["q"])  # D vem positivo e subtrai
inicial = clientes.dropna(subset=["cpf"]).groupby("cpf")["pontos_num"].sum()
ordem = pontos[pontos["cpf"].isin(inicial.index)].sort_values("criado_em")
menor = ordem.groupby("cpf")["mov"].cumsum().groupby(ordem["cpf"]).min()
print("negativos em algum momento | só banco:", (menor < 0).sum(), "| planilha + banco:", (menor + inicial.reindex(menor.index) < 0).sum())
print("saldo da planilha (CPF válido):", inicial.sum(), "| banco líquido:", pontos["mov"].sum())

In [ ]:
# saldos abreviados ("2,7k") e o teto de 15.000
abreviados = clientes["Pontos"].fillna("").str.lower().str.endswith("k")
print("abreviados:", abreviados.sum(), "| erro máximo: ±", abreviados.sum() * 50, "pts")
pd.cut(clientes["pontos_num"], [12999, 13999, 14999, 15000, float("inf")]).value_counts().sort_index()

In [ ]:
# clientes antigos: nas duas abas (178) e só na antiga (122)
antigos["p2023"] = pd.to_numeric(antigos["PONTOS_2023"])
saldo_2025 = clientes.dropna(subset=["Pontos"]).groupby("cpf")["pontos_num"].sum()  # Pontos vazio fica de fora, não vira 0
nas_duas = antigos[antigos["CPF/CNPJ"].isin(clientes["cpf"])].assign(p2025=lambda d: d["CPF/CNPJ"].map(saldo_2025))
print("nas duas abas:", len(nas_duas), "| 2023:", nas_duas["p2023"].sum(), "| 2025:", nas_duas["p2025"].sum(),
      "| caiu:", (nas_duas["p2025"] < nas_duas["p2023"]).sum(), "| subiu:", (nas_duas["p2025"] > nas_duas["p2023"]).sum(),
      "| vazio em 2025:", nas_duas["p2025"].isna().sum())
# a queda não segue a data da última compra (se fosse vencimento, quem comprou antes perderia mais)
display((nas_duas["p2025"] / nas_duas["p2023"]).groupby(pd.to_datetime(nas_duas["ULT_COMPRA"]).dt.quarter).median())
so_antigos = antigos[~antigos["CPF/CNPJ"].isin(clientes["cpf"])]
voltaram = pontos[pontos["cpf"].isin(so_antigos["CPF/CNPJ"])]
print("só na antiga:", len(so_antigos), "| pontos:", so_antigos["p2023"].sum(),
      "| ATIVO:", (so_antigos["STATUS"] == "ATIVO").sum(), so_antigos.loc[so_antigos["STATUS"] == "ATIVO", "p2023"].sum(),
      "| voltaram em 2025-26:", voltaram["cpf"].nunique(), "| resgataram:", voltaram.loc[voltaram["tipo"] == "D", "cpf"].nunique())

In [ ]:
# cadastrados depois do congelamento, mas com pontos na planilha
depois = clientes[(pd.to_datetime(clientes["Data Cadastro"], errors="coerce") >= "2025-01-01") & (clientes["pontos_num"] > 0)]
online = depois["loja_cadastro"].str.lower().isin(["app", "site"])
compras_deles = compras[compras["cliente_ref"].isin(depois["cpf"])]
print("cadastrados depois de 2025 com pontos:", len(depois), "| pontos:", depois["pontos_num"].sum(), "| pelo site ou app:", online.sum(),
      "| compras sem crédito:", (~compras_deles["id"].isin(pontos["compra_id"].dropna().str.removesuffix(".0"))).sum())

In [ ]:
# saldo negativo mesmo com o saldo inicial: volta logo, volta depois ou nunca volta
ordem = ordem.assign(saldo=ordem.groupby("cpf")["mov"].cumsum() + ordem["cpf"].map(inicial), quando=pd.to_datetime(ordem["criado_em"]))

def recuperacao(d):
    primeiro_negativo = d.loc[d["saldo"] < 0, "quando"].iloc[0]
    volta = d[(d["quando"] > primeiro_negativo) & (d["saldo"] >= 0)]
    if volta.empty:
        return "nunca volta"
    return "volta em menos de 1h" if volta["quando"].iloc[0] - primeiro_negativo < pd.Timedelta(hours=1) else "volta depois"

negativos = ordem[ordem["cpf"].isin(ordem.loc[ordem["saldo"] < 0, "cpf"])]
grupos = pd.Series({cpf: recuperacao(d) for cpf, d in negativos.groupby("cpf")})
print(grupos.value_counts().to_dict(), "| saldo final dos que nunca voltam:",
      negativos[negativos["cpf"].isin(grupos[grupos == "nunca volta"].index)].groupby("cpf")["saldo"].last().sum())

## Anotações para o dicionário de dados

**O que é cada linha**
- `clientes` (2.535): um **cadastro** do Vulto+, não uma pessoa: 131 CPFs têm 2 ou mais cadastros (ex.: a mesma Júlia com 85 pontos num e 246 no outro); 22 e-mails repetidos.
- `antigos` (300): cliente do programa de 2023. 178 também estão na aba Clientes. Pela aba "LEIA ISSO", os ATIVOS foram migrados em jan/24.
- `resgates` (596): resgate anotado na planilha, de 01/2023 a 09/2026.
- `pontos` (12.127): movimento no banco, de 01/01/2025 a 01/10/2026. C crédito (11.400), D resgate (565), E estorno (101), AJ ajuste (61).
- `compras` (14.948): compra que pode gerar pontos. Inclui as **vendas das lojas físicas** (origem `pdv`, 7.283), que não aparecem em `vendas.compra_id`: é por aqui que a loja física gera pontos.

**Colunas de `clientes`**

| Coluna | O que é | Problemas vistos |
|---|---|---|
| `cpf` | CPF limpo | 28 vazios (dígito verificador errado); 131 CPFs repetidos |
| `Pontos` | saldo congelado em 01/01/2025 | formatos `2.203`, `1389 pts`, `2,7k`, `15,0k`; 542 vazios; 26 cadastrados depois de 2025 com pontos |
| `categoria` | Bronze / Prata / Ouro | 14 grafias (`brz`, `Silver`, `prata*`, `Gold`, `ouro ★`, `vip?`); 425 vazias; "colocada na mão" |
| `aceita_msg` | aceita receber mensagem (LGPD) | 7 formas (`x`, `sim`, `1`, `S` / `nao`, `N`, `0`); 827 vazios |
| `loja_cadastro` | onde se cadastrou | 18 grafias |
| `Data Cadastro` | data do cadastro | 206 só com mês/ano; de 2021 a 2026 |
| `Tel` | telefone | 425 vazios (sem DDD) |

**Regras do Vulto+ que o sistema não aplica**
- **Dobro no online:** só os canais `site` e `app` ganharam 2 pontos por real. `web`, `e-commerce`, `ecommerce` e `online` ganharam 1 (ver o cruzamento canal × pontos por real).
- **Ouro +50%:** nunca aplicado.
- **Vencimento** (12 meses; 18 para Ouro): nunca aplicado.
- **Valor do ponto:** o banco usa R$ 0,10; a regra desde mar/2025 é R$ 0,05.
- **Saldo negativo:** 51 CPFs com saldo negativo só pelo banco.

**Bug dos pontos do app**
- **Fato:** entre 08 e 11/06/2026, 15 créditos do app deram pontos em dobro sobre o valor **em centavos**: 1.001.610 pontos. Entre 20 e 23/06/2026, 21 ajustes "correcao bug app v3.4" tiraram 1.138.676 pontos de 20 clientes.
- 11 clientes inflados foram corrigidos; **4 continuam inflados (215.982 pontos)**; 9 clientes levaram a correção sem crédito inflado visível.

**Resgates na planilha × banco**
- Desde 2025: 536 resgates na planilha (484 com CPF, 52 só com nome). Os 239 do "Site" estão todos no banco (o próprio sistema anota na planilha). Nas lojas físicas, 204 de 245 também estão no banco; ~41 aparecem só na planilha.
- **Risco:** descontar os resgates da planilha e do banco conta o mesmo resgate duas vezes.

**Clientes que só existem no banco**
- 534 CPFs têm movimentos de pontos no banco, mas não estão na aba Clientes (cadastrados depois do congelamento? CPF digitado errado?).

**Avisos (outras frentes)**
- **→ Vendas:** os 76 pedidos do app repetidos geraram 2 `compra_id` cada (pontos em dobro). Ver `01_vendas`.
- **→ Vendas:** a loja física gera pontos pela tabela `compras` (origem `pdv`), sem ligação direta com `vendas`.

**Problemas de clientes e pontos (lista única, nada decidido)**

| # | Problema | Quantos | Afeta |
|---|---|---|---|
| 1 | Mesma pessoa em vários cadastros | 131 CPFs | clientes únicos; soma de pontos |
| 2 | Saldo = planilha congelada + banco; formatos de pontos | 2.535 | passivo de pontos |
| 3 | Dobro no online aplicado só em `site` e `app` | ~2.100 créditos | passivo (a empresa deve mais?) |
| 4 | Ouro +50% e vencimento nunca aplicados | todos | passivo |
| 5 | Valor do ponto: R$ 0,10 × R$ 0,05 | — | passivo em R$ (metade ou o dobro) |
| 6 | Bug do app: 4 clientes ainda inflados (216 mil pontos) | 4 + 9 | passivo |
| 7 | Resgates só na planilha × duplicados com o banco | ~41 + 52 sem CPF | passivo |
| 8 | Clientes antigos também na aba nova | 178 | pontos contados duas vezes? |
| 9 | Categoria colocada na mão, 14 grafias | todos | regra do Ouro |
| 10 | `aceita_msg` vazio ou ambíguo | 827 | LGPD |
| 11 | CPFs com pontos no banco que não estão na planilha | 534 | clientes sem cadastro, mas com saldo |

**Desafios da revisão**
- **CPFs com 2+ cadastros:** dos 131, em 60 nenhum cadastro tem pontos e em 71 **os dois (ou mais) têm pontos**; nunca só um. Somar os cadastros extras acrescenta 49.973 pontos. Juntar os cadastros e somar os pontos ou ficar com um só é decisão de negócio (afeta o passivo).
- **CPFs só no banco (534):** nenhum difere em só 1 dígito de um CPF da planilha, então não parecem erros de digitação. 350 começaram em 2025 e 184 em 2026: **parecem clientes que passaram a ganhar pontos sem entrar na planilha** (que ficou congelada como fonte de pontos desde 01/01/2025). Eles têm saldo, mas não têm nome, e-mail nem telefone.

**Saldo inicial (K5)**
- **Planilha + banco:** o sistema somava a coluna `Pontos` aos movimentos do banco (`calcula_saldo` em `main.py`). Só com o banco, 183 clientes ficariam negativos em algum momento; com o saldo da planilha, 22.
- **Abreviados:** 96 saldos como `2,7k` (erro máximo de ±4.800 pts). **Teto:** 44 clientes com exatamente 15.000 e nenhum acima; nenhuma regra nem o código citam o limite.
- **Duas abas (178):** os saldos nunca são iguais (127 caíram, 47 subiram; 847.247 pts em 2023 × 504.217 em 2025). O sistema só lia a aba `Clientes`. Não há dado de 2024 em nenhuma fonte, e a queda não segue a data da última compra.
- **Só na aba antiga (122):** 466.627 pts, última compra em 2023; 31 ATIVOS (112.103 pts). 24 voltaram a comprar em 2025–26, começaram do zero e nenhum resgatou.
- **Cadastrados depois de 2025 com pontos (26, 7.352 pts):** 10 pelo site ou app; todas as compras dos outros ganharam pontos no banco. Origem não explicada.
- **Negativos (22):** 7 voltam em menos de 1 hora, 9 voltam depois e 6 nunca voltam (−2.547 pts). O `/resgate` não checa o saldo.

*(continuar preenchendo ao longo da análise)*